In [1]:
import numpy as np

# Rounding and Cancellation

## I Rounding errors
A number system can be represented as $\pm 1.b_1b_2b_3b_4 \times 2^m$ for $m \in [-6,6]$ and $b_i \in \{0,1\}$.  

Let’s say you want to represent the decimal number $19.625$ using the binary number system above. Can you represent this number exactly?  

$(19.625)_{10} = (10011.101)_{2} = (1.0011101)_{2} \times 2^{4}$  

$(1.0011)_{2} \times 2^{4} = 19$  
$(1.0100)_{2} \times 2^{4} = 20$  

### 1. Machine floating point number

• Not all real numbers can be exactly represented as a machine floating-point number.  

• Consider a real number in the normalized floating-point form: $$x = \pm 1.b_1b_2b_3...b_n\cdots \times 2^m$$

• The real number $x$ will be approximated by either $x_{-}$ or $x_{+}$, the nearest two machine floating point number to $x$.  

$$ x_{-} = (1.b_1b_2b_3...b_n)_{2} \times 2^m $$
$$ x_{+} = x_{-} + (0.\underbrace{00\cdots0}_{n-1\text{ zeros}}1)_2 \times 2^m = x_{-} + 2^{-n} \times 2^m = x_{-} + \epsilon _m \times 2^m$$
$$\Rightarrow \boxed{\text{gap} = \epsilon _m \times 2^m}$$

### 2. Rounding  
The process of replacing $x$ by a nearby machine number is called rounding, and the error involved is called roundoff error.  

**Round by chopping**:
|  | $x$ is positive | $x$ is negative |
| --- | --- | --- |
| Round up (ceil) | $fl(x) = x_{+}$ <br> rounding towards $+\infty$ | $fl(x) = x_{-}$ <br> rounding towards zero |
| Round down (floor) | $fl(x) = x_{-}$ <br> rounding towards zero | $fl(x) = x_{+}$ <br> rounding towards $+\infty$ |
  
**Round to nearest**: either round up or round down, whichever is closer

### 3. Rounding (roundoff) errors  

Consider rounding by chopping:

• **Absolute error**: $|fl(x) - x| \leq \epsilon _m \times 2^m$

• **Relative error**: $\frac{|fl(x) - x|}{|x|} \leq \frac{\epsilon _m \times 2^m}{|(1.b_1b_2b_3...b_n)_{2} \times 2^m |} \leq \frac{\epsilon _m}{|(1.b_1b_2b_3...b_n)_{2}|} \leq \boxed{\epsilon _m}$  

The relative error due to rounding (the process of representing a real number as a machine number) is always **bounded by machine epsilon**.

IEEE Single Precision: $\epsilon _m = 2^{-23} \approx 1.2 \times 10^{-7}$

$$e_r \leq 1.2 \times 10^{-7} \leq 5 \times 10^{-7} \Rightarrow 7 \text{ digits of precision}$$

IEEE Double Precision: $\epsilon _m = 2^{-52} \approx 2.2 \times 10^{-16}$

$$e_r \leq 2.2 \times 10^{-16} \leq 5 \times 10^{-7} \Rightarrow 16 \text{ digits of precision}$$


### 4. Gap between two machine numbers

Suppose we have a number $\delta$, what is the greatest $\delta$ such that we have $fl(x + \delta) = fl(x)$?  

$\Rightarrow \delta \leq \text{gap} = \boxed{\epsilon_m \times 2^m}$

Take $x=2^8$, single precision,  
Single: $\delta \leq \epsilon_m \times 2^8 < 2^{-23} \times 2^8 = 2^{-15}$  

$x=4.5 \times 10^4$,  
Double: $\delta \leq \epsilon_m \times 10^4 < 10^{-16} \times 10^4 = 10^{-12}$


## II Arithmetic with machine numbers  

### 1. Mathematical properties of FP operations

**Not necessarily associative**:  
For some $x,y,z$ the result below is possible: $$x+(y+z) \neq (x+y)+z$$

In [2]:
a = (np.pi+1e100)-1e100
b = np.pi+(1e100-1e100)
print(a)
print(b)

0.0
3.141592653589793


**Not necessarily distributive**:   
For some $x,y,z$ the result below is possible: $$z(x+y) \neq z(x)+z(y)$$

In [3]:
print(100*(0.1+0.2))
print(100*0.1+100*0.2)

30.000000000000004
30.0


**Not necessarily cumulative**:   
Repeatedly adding a very small number to a large number may do nothing

In [6]:
a = 1e50
for i in range(10):
    print(a + np.pi)

1e+50
1e+50
1e+50
1e+50
1e+50
1e+50
1e+50
1e+50
1e+50
1e+50


### 2. Floating point arithmetic (basic idea)

$$x = (-1)^s \times 1.f \times 2^m$$

• First compute the exact result  
• Then round the result to make it fit into the desired precision  

• $x + y = fl(x + y)$  

• $x \times y = fl(x \times y)$ 

Consider a number system such that $\pm 1.b_1b_2 \times 2^m$ for $m \in [-4,4]$ and $b_i \in \{0,1\}$.  
Rough algorithm for addition and subtraction:
1. Bring both numbers onto a common exponent
2. Do “grade-school” operation
3. Round result

• Example 1: No rounding needed  
$x_1 = (1.101)_2 \times 2^1$,   $x_2 = (1.001)_2 \times 2^1$
$$c = x_1 + x_2 =  (10.110)_2 \times 2^1 = (1.011)_2 \times 2^2$$

• Example 2: Require rounding  
$x_1 = (1.101)_2 \times 2^0$,   $x_2 = (1.000)_2 \times 2^0$
$$c = x_1 + x_2 =  (10.101)_2 \times 2^0 = (1.0101)_2 \times 2^1 \Rightarrow fl(c) = 1.010 \times 2^1$$

• Example 3:  
$x_1 = (1.100)_2 \times 2^1$,   $x_2 = (1.101)_2 \times 2^{-1}$
$$c = x_1 + x_2 = (1.100)_2 \times 2^1 + (0.011)_2 \times 2^{1} = (1.111)_2 \times 2^1$$
The last two bits of $x_2$ are dropped in the addition.


**Catastrophic Cancellation**  

• Example 4:  
$x_1 = (1.111)_2 \times 2^1$,   $x_2 = (1.110)_2 \times 2^{1}$
$$c = x_1 - x_2 = (1.111)_2 \times 2^1 - (1.110)_2 \times 2^{1} = (0.001)_2 \times 2^1 = (1.???)_2 \times 2^3 \Rightarrow fl(c) = (1.000)_2 \times 2^3$$

• There is not data to indicate what the missing digits should be.  
• Machine fills them with its best guess, which is often not good (usually what is called spurious zeros).  
• Number of significant digits in the result is reduced($p = 4 \Rightarrow p = 1$).  



### 3. Loss of significance  

Assume $a$ and $b$ are real numbers with $a \gg b$. For example,   
$a = 1.a_1a_2\cdots a_n \cdots \times 2^0$  
$b = 1.b_1b_2\cdots b_n \cdots \times 2^{-8}$  
In Single Precision (n=23), compute $a + b$

$a + b = 1.a_1a_2\cdots a_{23} \times 2^0 + 0.\underbrace{00\cdots0}_{7\text{ zeros}}1b_1\cdots b_{14}b_{15} \times 2^0$  
$\Rightarrow fl(a+b)$ has only 15 significant digits of $b$


### 4. Cancellation

Assume $a$ and $b$ are real numbers with $a \approx b$. For example,   
$a = 1.a_1a_2\cdots a_n \cdots \times 2^m$  
$b = 1.b_1b_2\cdots b_n \cdots \times 2^m$  

In single precision (without loss of generality), consider this example:  

$a = 1.a_1a_2\cdots a_{21}11a_{24}a_{25}a_{26}\cdots \times 2^m$     
$b = 1.a_1a_2\cdots a_{21}10b_{24}b_{25}b_{26}\cdots \times 2^m$   
$a - b = 0.\underbrace{00\cdots 0}_{22 \text{ zeros}}1 \times 2^m$  
$fl(a-b) = 0.\underbrace{00\cdots 0}_{22 \text{ zeros}}1 \times 2^m = 1.\underbrace{00\cdots 0}_{\text{not sig.}} \times 2^{-23} \times 2^m$

How can we avoid this loss of significance? For example, consider the function $f(x) = \sqrt{x^2 + 1} - 1$  
If we want to evaluate the function for values ! near zero, there is a potential loss of significance in the subtraction.  
Assume you are performing this computation using a machine with 5  decimal accurate digits. Compute $f(10^{-3})$

$1.000000 + 0.000001 = 1.000001 = 1.00000$ under this rounding scheme.  
$\Rightarrow f(10^{-3}) = \sqrt{10^{-6} + 1} - 1 = 1 - 1 = 0$  

Re-write the function $f(x) = \sqrt{x^2 + 1} - 1$ to avoid subtraction of two numbers with similar order of magnitude:  use $(a+b)(a-b) = a^2 - b^2$  
$\Rightarrow f(x) = \frac{x^2 + 1 - 1}{\sqrt{x^2 + 1} + 1} = \frac{x^2}{\sqrt{x^2 + 1} + 1}$  
$\Rightarrow f(10^{-3}) = \frac{10^{-6}}{\sqrt{10^{-6} + 1} + 1} = \frac{10^{-6}}{2}$  


<details>
<summary>Example</summary>

If $x = 3.1415276549$ and $y = 3.1402143526$, what is the relative error in the computation of $x - y$ in a computer with $4$ decimal digits of accuracy?

<details>
<summary>Solution</summary>

$fl(x) = 3.141$  
$fl(y) = 3.140$  
$fl(x - y) = 0.001$  
$e_r = \frac{|fl(x - y) - (x - y)|}{|x - y|} = \frac{0.0013133023 - 0.001}{0.0013133023} \approx \boxed{0.2386}$
</details>